In [ ]:

import duckdb
import pandas as pd
import glob

# ============================================================
# 1. INITIALIZE DUCKDB
# ============================================================

print("Initializing DuckDB...")

con = duckdb.connect("bikeshare_dataset.duckdb")

con.execute("SET memory_limit = '4GB';")
con.execute("SET preserve_insertion_order = false;")


# ============================================================
# 2. BUILD MASTER STATIONS TABLE
# ============================================================

print("Processing station coordinates...")

stations_glob = (
    "C:/[TODO]/Dataset/Pre 2020/Stations/*.csv"
)

con.execute(f"""
    CREATE OR REPLACE TABLE master_stations AS
    SELECT DISTINCT ON (id)
        id,
        latitude,
        longitude
    FROM read_csv_auto(
        '{stations_glob}',
        union_by_name=True,
        all_varchar=True
    )
    WHERE id IS NOT NULL
      AND latitude IS NOT NULL;
""")


# ============================================================
# 3. CREATE UNIFIED TARGET TABLE
# ============================================================

print("Creating unified trips table...")

con.execute("""
    CREATE OR REPLACE TABLE final_harmonized_trips (
        ride_id VARCHAR,
        rideable_type VARCHAR,
        started_at VARCHAR,
        ended_at VARCHAR,
        start_station_name VARCHAR,
        start_station_id VARCHAR,
        end_station_name VARCHAR,
        end_station_id VARCHAR,
        start_lat DOUBLE,
        start_lng DOUBLE,
        end_lat DOUBLE,
        end_lng DOUBLE,
        member_casual VARCHAR
    );
""")


# ============================================================
# 4. PROCESS PRE-2020 HISTORICAL DATA
# ============================================================

def process_pre_2020(file_pattern):

    files = glob.glob(file_pattern)

    for f in files:

        print(f"Processing pre-2020 file: {f}")

        df = pd.read_csv(f, dtype=str, low_memory=False)

        # Standardize varying historical column names
        col_map = {}

        for col in df.columns:

            c_lower = col.strip().lower()

            if "trip_id" in c_lower or "rental id" in c_lower:
                col_map[col] = "ride_id"

            elif c_lower in [
                "starttime",
                "start_time",
                "01 - rental details local start time"
            ]:
                col_map[col] = "started_at"

            elif c_lower in [
                "stoptime",
                "end_time",
                "01 - rental details local end time"
            ]:
                col_map[col] = "ended_at"

            elif c_lower in [
                "from_station_name",
                "03 - rental start station name"
            ]:
                col_map[col] = "start_station_name"

            elif c_lower in [
                "from_station_id",
                "03 - rental start station id"
            ]:
                col_map[col] = "start_station_id"

            elif c_lower in [
                "to_station_name",
                "02 - rental end station name"
            ]:
                col_map[col] = "end_station_name"

            elif c_lower in [
                "to_station_id",
                "02 - rental end station id"
            ]:
                col_map[col] = "end_station_id"

            elif c_lower in ["usertype", "user type"]:
                col_map[col] = "member_casual"

        df = df.rename(columns=col_map)

        # Ensure all required columns exist
        required = [
            "ride_id",
            "started_at",
            "ended_at",
            "start_station_name",
            "start_station_id",
            "end_station_name",
            "end_station_id",
            "member_casual"
        ]

        for col in required:
            if col not in df.columns:
                df[col] = None

        # Historical data uses classic bikes
        df["rideable_type"] = "classic_bike"

        # Standardize membership categories
        df["member_casual"] = df["member_casual"].replace({
            "Subscriber": "member",
            "subscriber": "member",
            "Customer": "casual",
            "customer": "casual"
        })

        # Register dataframe in DuckDB
        con.register("df_view", df)

        # Append historical trips and join station coordinates
        con.execute("""
            INSERT INTO final_harmonized_trips

            SELECT
                t.ride_id,
                t.rideable_type,
                t.started_at,
                t.ended_at,
                t.start_station_name,
                t.start_station_id,
                t.end_station_name,
                t.end_station_id,
                TRY_CAST(s1.latitude AS DOUBLE),
                TRY_CAST(s1.longitude AS DOUBLE),
                TRY_CAST(s2.latitude AS DOUBLE),
                TRY_CAST(s2.longitude AS DOUBLE),
                t.member_casual

            FROM df_view t

            LEFT JOIN master_stations s1
                ON TRIM(t.start_station_id) = TRIM(s1.id)

            LEFT JOIN master_stations s2
                ON TRIM(t.end_station_id) = TRIM(s2.id);
        """)

        con.unregister("df_view")


print("\nStarting pre-2020 processing...")

process_pre_2020(
    "C:/[TODO]/Dataset/Pre 2020/Trips/*.csv"
)


# ============================================================
# 5. PROCESS POST-2020 DATA
# ============================================================

print("\nStarting post-2020 processing...")

post_2020_glob = (
    "C:/[TODO]/Dataset/Post 2020/*.csv"
)

# Read post-2020 CSVs with schema evolution support
con.execute(f"""
    INSERT INTO final_harmonized_trips

    SELECT
        CAST(ride_id AS VARCHAR),
        CAST(rideable_type AS VARCHAR),
        CAST(started_at AS VARCHAR),
        CAST(ended_at AS VARCHAR),
        CAST(start_station_name AS VARCHAR),
        CAST(start_station_id AS VARCHAR),
        CAST(end_station_name AS VARCHAR),
        CAST(end_station_id AS VARCHAR),
        TRY_CAST(start_lat AS DOUBLE),
        TRY_CAST(start_lng AS DOUBLE),
        TRY_CAST(end_lat AS DOUBLE),
        TRY_CAST(end_lng AS DOUBLE),
        LOWER(CAST(member_casual AS VARCHAR))

    FROM read_csv_auto(
        '{post_2020_glob}',
        union_by_name=True,
        all_varchar=True
    );
""")


# ============================================================
# 6. EXPORT EVERYTHING TO ONE PARQUET FILE
# ============================================================

print("\nExporting unified dataset...")

con.execute("""
    COPY final_harmonized_trips
    TO 'bikeshare_dataset.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")


# ============================================================
# 7. VERIFY RESULTS
# ============================================================

total_rows = con.execute("""
    SELECT COUNT(*)
    FROM final_harmonized_trips
""").fetchone()[0]

print("\n====================================")
print("SUCCESS! DATASETS COMBINED.")
print("====================================")

print(f"Total trips: {total_rows:,}")
print("DuckDB database: bikeshare_dataset.duckdb")
print("Parquet file: bikeshare_dataset.parquet")

# Show a sample
print("\nSample of combined data:")

print(
    con.execute("""
        SELECT *
        FROM final_harmonized_trips
        LIMIT 5
    """).df()
)

con.close()


Initializing DuckDB...
Processing station coordinates...
Creating unified trips table...

Starting pre-2020 processing...
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_2013.csv
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_2014-Q3-07.csv
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_2014-Q3-0809.csv
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_2014-Q4.csv
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_2014_Q1Q2.csv
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_2015-Q1.csv
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_2015-Q2.csv
Processing pre-2020 file: C:/Users/vanoi/Downloads/ADA Project/Dataset/Pre 2020/Trips\Divvy_Trips_201